# Arch-A: Alpha AGI — Kaggle 2×T4

**Purpose:** Validate the architecture is bug-free before real pre-training.

This notebook tests every architectural component: forward, backward, training stack, quantization, and saves a portable checkpoint for AMD.

## Cell 1 — Install & Hardware

In [ ]:
!pip install -q transformers einops
import torch, os, sys, time, gc
from pathlib import Path
print(f'PyTorch : {torch.__version__}')
n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Primary: {device}')
def vram():
    for i in range(torch.cuda.device_count()):
        a=torch.cuda.memory_allocated(i)/1024**3
        t=torch.cuda.get_device_properties(i).total_memory/1024**3
        print(f'  GPU {i}: {a:.2f}/{t:.1f} GiB ({100*a/t:.0f}%)')

## Cell 2 — Write All Source Files

In [ ]:
import os, sys
BASE = '/kaggle/working'
DIRS = ['arch_a','arch_a/modules','arch_a/training','arch_a/quantization','arch_a/kernels']
for d in DIRS: os.makedirs(os.path.join(BASE,d), exist_ok=True)
FILES = {}
FILES["arch_a/__init__.py"] = "\"\"\"Arch-A: Alpha AGI research implementation.\"\"\"\nfrom .config import ArchAConfig\nfrom .model import ArchAForCausalLM, ArchAModel, ArchAOutput\n__version__ = \"0.3.2\"\n__all__ = [\"ArchAConfig\", \"ArchAForCausalLM\", \"ArchAModel\", \"ArchAOutput\"]\n"
FILES["arch_a/config.py"] = "\nfrom __future__ import annotations\nfrom dataclasses import dataclass, asdict\nfrom typing import Any, Dict, Optional\n\n@dataclass\nclass ArchAConfig:\n    vocab_size: int = 65536\n    d_model: int = 768\n    n_layers: int = 12\n    n_heads: int = 12\n    n_kv_heads: int = 4\n    d_head: int = 64\n    d_ff: int = 3072\n    ssm_state_dim: int = 128\n    alpha_window: int = 256\n    max_seq_len: int = 2048\n    rope_theta: float = 10000.0\n    dropout: float = 0.0\n    attn_dropout: float = 0.0\n    norm_eps: float = 1e-5\n    algr_max_loops: int = 3\n    algr_confidence_threshold: float = 0.82\n    algr_temperature: float = 1.0\n    nadd_steps: int = 6\n    nadd_hidden_mult: int = 2\n    nadd_noise_scale: float = 0.15\n    use_bias: bool = False\n    use_checkpointing: bool = True\n    tie_embeddings: bool = True\n    use_fp32_residuals: bool = True\n    use_structured_memory: bool = True\n    device_map: Optional[Dict[int, str]] = None  # layer shard plan if provided\n    quant_block_size: int = 32\n    shadow_bits: int = 4\n    polar_bits: int = 4\n    scale_critical_lr: float = 1e-4\n    scale_noncritical_lr: float = 3e-4\n    galore_rank: int = 8\n    spectron_power_iters: int = 1\n    spectron_target_norm: float = 1.0\n    residual_summary_dim: int = 128\n    num_codebooks: int = 1\n    model_name: str = \"arch_a\"\n\n    @classmethod\n    def for_debug(cls) -> \"ArchAConfig\":\n        return cls(\n            vocab_size=512, d_model=64, n_layers=1, n_heads=4, n_kv_heads=2,\n            d_head=16, d_ff=256, ssm_state_dim=32, alpha_window=32, max_seq_len=128,\n            nadd_steps=2, algr_max_loops=1\n        )\n\n    @classmethod\n    def for_2gpu_demo(cls) -> \"ArchAConfig\":\n        return cls(\n            vocab_size=1024, d_model=64, n_layers=2, n_heads=4, n_kv_heads=2,\n            d_head=16, d_ff=256, ssm_state_dim=32, alpha_window=32, max_seq_len=128,\n            nadd_steps=2, algr_max_loops=1\n        )\n\n    @classmethod\n    def for_500m(cls) -> \"ArchAConfig\":        return cls(\n            vocab_size=32768, d_model=1024, n_layers=16, n_heads=16, n_kv_heads=4,\n            d_head=64, d_ff=4096, ssm_state_dim=128, alpha_window=256, max_seq_len=2048,\n            nadd_steps=6, algr_max_loops=3\n        )\n\n    @classmethod\n    def for_1b(cls) -> \"ArchAConfig\":\n        return cls(\n            vocab_size=65536, d_model=1536, n_layers=24, n_heads=24, n_kv_heads=6,\n            d_head=64, d_ff=6144, ssm_state_dim=192, alpha_window=256, max_seq_len=4096,\n            nadd_steps=8, algr_max_loops=3\n        )\n\n    def to_dict(self) -> Dict[str, Any]:\n        return asdict(self)\n\n    @classmethod\n    def from_dict(cls, d: Dict[str, Any]) -> \"ArchAConfig\":\n        return cls(**d)\n"
FILES["arch_a/modules/__init__.py"] = "\nfrom .normalization import RMSNorm\nfrom .rotary import RotaryEmbedding, apply_rope\nfrom .alpha_window import AlphaWindow\nfrom .algr import ALGRBlock, ALGRController\nfrom .nadd import NADDDecoder, semantic_global_anchor\n"
FILES["arch_a/modules/normalization.py"] = "\nfrom __future__ import annotations\nimport torch\nfrom torch import nn\n\nclass RMSNorm(nn.Module):\n    def __init__(self, d_model: int, eps: float = 1e-5):\n        super().__init__()\n        self.eps = eps\n        self.weight = nn.Parameter(torch.ones(d_model))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        orig_dtype = x.dtype\n        x_fp32 = x.float()\n        rms = torch.rsqrt(x_fp32.pow(2).mean(dim=-1, keepdim=True) + self.eps)\n        out = x_fp32 * rms * self.weight.float()\n        return out.to(orig_dtype)\n"
FILES["arch_a/modules/rotary.py"] = "\nfrom __future__ import annotations\nimport torch\nfrom torch import nn\n\ndef _rotate_half(x: torch.Tensor) -> torch.Tensor:\n    x1 = x[..., ::2]\n    x2 = x[..., 1::2]\n    out = torch.stack((-x2, x1), dim=-1)\n    return out.flatten(-2)\n\nclass RotaryEmbedding(nn.Module):\n    def __init__(self, dim: int, base: float = 10000.0, max_seq_len: int = 2048):\n        super().__init__()\n        self.dim = dim\n        self.base = base\n        self.max_seq_len = max_seq_len\n        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))\n        self.register_buffer(\"inv_freq\", inv_freq, persistent=False)\n\n    def get_cos_sin(self, seq_len: int, device, dtype):\n        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)\n        freqs = torch.einsum(\"i,j->ij\", t, self.inv_freq)\n        emb = torch.cat((freqs, freqs), dim=-1)\n        return emb.cos().to(dtype), emb.sin().to(dtype)\n\ndef apply_rope(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):\n    q_out = (q * cos) + (_rotate_half(q) * sin)\n    k_out = (k * cos) + (_rotate_half(k) * sin)\n    return q_out, k_out\n"
FILES["arch_a/modules/alpha_window.py"] = "from __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import Optional, Tuple\nimport math\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom .normalization import RMSNorm\nfrom .rotary import RotaryEmbedding, apply_rope\n\nAlphaWindowState = Tuple[torch.Tensor, torch.Tensor]  # (ssm_state, summary_state)\n\nclass AlphaWindow(nn.Module):\n    \"\"\"\n    Hybrid SSM + attention block with explicit dtype discipline.\n    The SSM is computed in fp32 for stability and cast back to the module dtype.\n    \"\"\"\n    def __init__(\n        self,\n        d_model: int,\n        n_heads: int,\n        n_kv_heads: int,\n        d_head: int,\n        ssm_state_dim: int,\n        alpha_window: int,\n        rope_theta: float = 10000.0,\n        norm_eps: float = 1e-5,\n        attn_dropout: float = 0.0,\n        use_bias: bool = False,\n        residual_fp32: bool = True,\n        summary_dim: int = 128,\n    ):\n        super().__init__()\n        self.d_model = d_model\n        self.n_heads = n_heads\n        self.n_kv_heads = n_kv_heads\n        self.d_head = d_head\n        self.ssm_state_dim = ssm_state_dim\n        self.alpha_window = alpha_window\n        self.residual_fp32 = residual_fp32\n        self.summary_dim = summary_dim\n\n        assert d_model == n_heads * d_head, \"d_model must equal n_heads * d_head\"\n        assert n_heads % n_kv_heads == 0, \"n_heads must be divisible by n_kv_heads\"\n        self.n_rep = n_heads // n_kv_heads\n\n        self.norm1 = RMSNorm(d_model, norm_eps)\n        self.norm_ssm = RMSNorm(d_model, norm_eps)\n        self.norm_attn = RMSNorm(d_model, norm_eps)\n\n        self.in_proj = nn.Linear(d_model, ssm_state_dim, bias=use_bias)\n        self.state_proj = nn.Linear(ssm_state_dim, d_model, bias=use_bias)\n        self.decay = nn.Parameter(torch.zeros(ssm_state_dim))\n        self.gate = nn.Linear(d_model, d_model, bias=use_bias)\n        self.summary_proj = nn.Linear(d_model, summary_dim, bias=use_bias)\n        self.summary_to_model = nn.Linear(summary_dim, d_model, bias=use_bias)\n\n        self.q_proj = nn.Linear(d_model, n_heads * d_head, bias=use_bias)\n        self.k_proj = nn.Linear(d_model, n_kv_heads * d_head, bias=use_bias)\n        self.v_proj = nn.Linear(d_model, n_kv_heads * d_head, bias=use_bias)\n        self.o_proj = nn.Linear(n_heads * d_head, d_model, bias=use_bias)\n\n        self.rotary = RotaryEmbedding(d_head, base=rope_theta, max_seq_len=alpha_window)\n        self.attn_dropout = attn_dropout\n\n    def _init_state(self, x: torch.Tensor) -> AlphaWindowState:\n        bsz = x.size(0)\n        device = x.device\n        dtype = x.dtype\n        ssm_state = torch.zeros(bsz, self.ssm_state_dim, device=device, dtype=dtype)\n        summary_state = torch.zeros(bsz, self.summary_dim, device=device, dtype=dtype)\n        return ssm_state, summary_state\n\n    def _ssm_scan(self, x: torch.Tensor, ssm_state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:\n        # vectorized fp32 scan for stability and speed\n        x_fp32 = x.float()\n        h0 = ssm_state.float()\n        decay = torch.sigmoid(self.decay.float()).clamp(1e-4, 1 - 1e-4).view(1, -1)\n        in_w = self.in_proj.weight.float()\n        in_b = self.in_proj.bias.float() if self.in_proj.bias is not None else None\n        out_w = self.state_proj.weight.float()\n        out_b = self.state_proj.bias.float() if self.state_proj.bias is not None else None\n\n        u = torch.tanh(torch.nn.functional.linear(x_fp32, in_w, in_b))  # [B,T,S]\n        t = torch.arange(u.size(1), device=u.device, dtype=u.dtype).unsqueeze(-1)  # [T,1]\n        log_decay = torch.log(decay)\n        powers = torch.exp(t * log_decay)  # [T,S]\n        u_scaled = u / powers.unsqueeze(0)\n        h_seq = powers.unsqueeze(0) * (h0.unsqueeze(1) + torch.cumsum(u_scaled, dim=1))\n        y = torch.nn.functional.linear(h_seq, out_w, out_b)\n        h_last = h_seq[:, -1]\n        return y, h_last\n\n    def _local_attention(self, x: torch.Tensor) -> torch.Tensor:\n        bsz, seq_len, _ = x.shape\n        local_len = min(seq_len, self.alpha_window)\n        x_local = x[:, -local_len:]\n        q = self.q_proj(self.norm_attn(x_local))\n        k = self.k_proj(self.norm_attn(x_local))\n        v = self.v_proj(self.norm_attn(x_local))\n\n        q = q.view(bsz, local_len, self.n_heads, self.d_head).transpose(1, 2)  # B,H,T,D\n        k = k.view(bsz, local_len, self.n_kv_heads, self.d_head).transpose(1, 2)  # B,K,T,D\n        v = v.view(bsz, local_len, self.n_kv_heads, self.d_head).transpose(1, 2)\n\n        if self.n_rep > 1:\n            k = k.repeat_interleave(self.n_rep, dim=1)\n            v = v.repeat_interleave(self.n_rep, dim=1)\n\n        cos, sin = self.rotary.get_cos_sin(local_len, x.device, q.dtype)\n        cos = cos.view(1, 1, local_len, self.d_head)\n        sin = sin.view(1, 1, local_len, self.d_head)\n        q, k = apply_rope(q, k, cos, sin)\n\n        # causal mask\n        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)\n        mask = torch.ones(local_len, local_len, device=x.device, dtype=torch.bool).triu(1)\n        attn_scores = attn_scores.masked_fill(mask.view(1, 1, local_len, local_len), float(\"-inf\"))\n        attn_probs = torch.softmax(attn_scores.float(), dim=-1).to(q.dtype)\n        if self.attn_dropout and self.training:\n            attn_probs = F.dropout(attn_probs, p=self.attn_dropout)\n        out = torch.matmul(attn_probs, v)  # B,H,T,D\n        out = out.transpose(1, 2).contiguous().view(bsz, local_len, self.n_heads * self.d_head)\n        out = self.o_proj(out)\n        if local_len < seq_len:\n            pad = torch.zeros(bsz, seq_len - local_len, self.d_model, device=out.device, dtype=out.dtype)\n            out = torch.cat([pad, out], dim=1)\n        return out\n\n    def forward(\n        self,\n        x: torch.Tensor,\n        ssm_state: Optional[AlphaWindowState] = None,\n    ) -> Tuple[torch.Tensor, AlphaWindowState]:\n        if ssm_state is None:\n            ssm_state = self._init_state(x)\n        ssm_h, summary_h = ssm_state\n\n        orig_dtype = x.dtype\n        x_norm = self.norm1(x)\n        x_gate = torch.sigmoid(self.gate(x_norm))\n\n        ssm_out, new_ssm_h = self._ssm_scan(self.norm_ssm(x_norm), ssm_h)\n        attn_out = self._local_attention(x_norm)\n\n        # Fractal summary update (EMA over batch summary)\n        pooled = x_norm.float().mean(dim=1)\n        summary_update = torch.tanh(self.summary_proj(pooled)).to(orig_dtype)\n        summary_h = 0.97 * summary_h + 0.03 * summary_update\n        summary_bias = self.summary_to_model(summary_h).unsqueeze(1)\n\n        y = x_norm + x_gate * (ssm_out.to(orig_dtype) + attn_out + summary_bias)\n        y = y.to(self.o_proj.weight.dtype)\n\n        # keep residual branch dtype aligned\n        if self.residual_fp32:\n            y = y.float().to(orig_dtype)\n\n        return y, (new_ssm_h.to(orig_dtype), summary_h.to(orig_dtype))\n"
FILES["arch_a/modules/algr.py"] = "from __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import List, Optional, Sequence, Tuple\nimport torch\nfrom torch import nn\nfrom .alpha_window import AlphaWindow, AlphaWindowState\nfrom .normalization import RMSNorm\n\n@dataclass\nclass ALGRMeta:\n    loops: List[int]\n    halt_prob: List[float]\n    entropy: List[float]\n    confidence: List[float]\n\nclass ALGRBlock(nn.Module):\n    def __init__(\n        self,\n        d_model: int,\n        n_heads: int,\n        n_kv_heads: int,\n        d_head: int,\n        ssm_state_dim: int,\n        alpha_window: int,\n        d_ff: int,\n        rope_theta: float,\n        norm_eps: float,\n        dropout: float,\n        use_bias: bool,\n        residual_fp32: bool,\n        summary_dim: int,\n    ):\n        super().__init__()\n        self.alpha_window = AlphaWindow(\n            d_model=d_model,\n            n_heads=n_heads,\n            n_kv_heads=n_kv_heads,\n            d_head=d_head,\n            ssm_state_dim=ssm_state_dim,\n            alpha_window=alpha_window,\n            rope_theta=rope_theta,\n            norm_eps=norm_eps,\n            attn_dropout=dropout,\n            use_bias=use_bias,\n            residual_fp32=residual_fp32,\n            summary_dim=summary_dim,\n        )\n        self.norm2 = RMSNorm(d_model, norm_eps)\n        self.mlp = nn.Sequential(\n            nn.Linear(d_model, d_ff, bias=use_bias),\n            nn.SiLU(),\n            nn.Linear(d_ff, d_model, bias=use_bias),\n        )\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x: torch.Tensor, ssm_state: Optional[AlphaWindowState] = None, loop_idx: int = 0):\n        residual = x\n        x, new_state = self.alpha_window(x, ssm_state)\n        x = residual + self.dropout(x)\n        residual2 = x\n        x = self.mlp(self.norm2(x))\n        x = residual2 + self.dropout(x)\n        return x, new_state\n\nclass ALGRController(nn.Module):\n    \"\"\"\n    Adaptive Logic-Gated Recurrence controller.\n    The controller loops each block until a halting gate crosses a threshold.\n    \"\"\"\n    def __init__(\n        self,\n        layers: nn.ModuleList,\n        d_model: int,\n        max_loops: int = 3,\n        confidence_threshold: float = 0.82,\n        temperature: float = 1.0,\n        device_map: Optional[Sequence[torch.device]] = None,\n    ):\n        super().__init__()\n        self.layers = layers\n        self.max_loops = max_loops\n        self.confidence_threshold = confidence_threshold\n        self.temperature = temperature\n        self.device_map = list(device_map) if device_map is not None else None\n        self.halt_head = nn.Sequential(\n            nn.Linear(d_model, d_model // 2),\n            nn.SiLU(),\n            nn.Linear(d_model // 2, 1),\n        )\n\n    def _move_optional_state(self, state, device):\n        if state is None:\n            return None\n        if isinstance(state, tuple):\n            return tuple(s.to(device, non_blocking=True) if s is not None else None for s in state)\n        return state.to(device, non_blocking=True)\n\n    def forward(self, layers, x, ssm_states, training: bool = True):\n        if ssm_states is None:\n            ssm_states = [None] * len(layers)\n        meta_loops, meta_halt, meta_entropy, meta_conf = [], [], [], []\n\n        for i, layer in enumerate(layers):\n            dev = self.device_map[i] if self.device_map is not None else x.device\n            if x.device != dev:\n                x = x.to(dev, non_blocking=True)\n            ssm_states[i] = self._move_optional_state(ssm_states[i], dev)\n\n            # keep halting head on the same device as the activations\n            halt_dev = x.device\n            if next(self.halt_head.parameters()).device != halt_dev:\n                self.halt_head.to(halt_dev)\n\n            loops = 0\n            halted = False\n            conf = 0.0\n            entropy = 0.0\n            while True:\n                x, ssm_states[i] = layer(x, ssm_states[i], loop_idx=loops)\n                pooled = x.float().mean(dim=1)\n                halt_logit = self.halt_head(pooled).squeeze(-1) / max(self.temperature, 1e-6)\n                prob = torch.sigmoid(halt_logit)\n                conf = float(prob.mean().detach().cpu())\n                p = prob.clamp(1e-6, 1 - 1e-6)\n                ent = (-p * torch.log(p) - (1 - p) * torch.log(1 - p)).mean()\n                entropy = float(ent.detach().cpu())\n                loops += 1\n                if (prob >= self.confidence_threshold).all() or loops >= self.max_loops:\n                    halted = True\n                    break\n            meta_loops.append(loops)\n            meta_halt.append(float(halted))\n            meta_entropy.append(entropy)\n            meta_conf.append(conf)\n\n        return x, ssm_states, ALGRMeta(meta_loops, meta_halt, meta_entropy, meta_conf)\n"
FILES["arch_a/modules/nadd.py"] = "from __future__ import annotations\nfrom typing import Optional, Tuple\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom .normalization import RMSNorm\n\ndef semantic_global_anchor(logits: torch.Tensor, anchor: Optional[torch.Tensor] = None, strength: Optional[torch.Tensor] = None):\n    if anchor is None:\n        return logits\n    if anchor.dim() == logits.dim() - 1:\n        anchor = anchor.unsqueeze(1)\n    if strength is None:\n        return logits + anchor\n    while strength.dim() < logits.dim():\n        strength = strength.unsqueeze(-1)\n    return logits + strength * anchor\n\nclass NADDRefiner(nn.Module):\n    def __init__(self, d_model: int, hidden_mult: int = 2, dropout: float = 0.0, use_bias: bool = False):\n        super().__init__()\n        inner = d_model * hidden_mult\n        self.norm = RMSNorm(d_model)\n        self.net = nn.Sequential(\n            nn.Linear(d_model, inner, bias=use_bias),\n            nn.SiLU(),\n            nn.Dropout(dropout),\n            nn.Linear(inner, d_model, bias=use_bias),\n        )\n        self.gate = nn.Linear(d_model, d_model, bias=use_bias)\n\n    def forward(self, x: torch.Tensor):\n        x_norm = self.norm(x)\n        delta = self.net(x_norm)\n        g = torch.sigmoid(self.gate(x_norm))\n        return x + g * delta\n\nclass NADDDecoder(nn.Module):\n    \"\"\"\n    Non-autoregressive diffusion-style iterative refinement decoder.\n    Works as a denoising latent refinement stack, not a discrete diffusion solver.\n    \"\"\"\n    def __init__(\n        self,\n        d_model: int,\n        vocab_size: int,\n        steps: int = 6,\n        hidden_mult: int = 2,\n        dropout: float = 0.0,\n        use_bias: bool = False,\n        noise_scale: float = 0.15,\n    ):\n        super().__init__()\n        self.steps = steps\n        self.noise_scale = noise_scale\n        self.refiners = nn.ModuleList([NADDRefiner(d_model, hidden_mult, dropout, use_bias) for _ in range(steps)])\n        self.out_norm = RMSNorm(d_model)\n        self.out_proj = nn.Linear(d_model, vocab_size, bias=False)\n        self.anchor_proj = nn.Linear(d_model, vocab_size, bias=use_bias)\n\n    def corrupt(self, x: torch.Tensor):\n        if not self.training or self.noise_scale <= 0:\n            return x\n        noise = torch.randn_like(x) * self.noise_scale\n        return x + noise\n\n    def forward(self, x: torch.Tensor, anchor_state: Optional[torch.Tensor] = None):\n        x = x.to(self.out_proj.weight.dtype)\n        x = self.corrupt(x)\n        for refiner in self.refiners:\n            x = refiner(x)\n        logits = self.out_proj(self.out_norm(x))\n        if anchor_state is not None:\n            anchor_state = anchor_state.to(x.dtype)\n            anchor = self.anchor_proj(anchor_state).unsqueeze(1)  # [B,1,V]\n            strength = torch.sigmoid(anchor_state.float().mean(dim=-1, keepdim=True)).unsqueeze(-1)  # [B,1,1]\n            logits = semantic_global_anchor(logits, anchor, strength)\n        return x, logits\n"
FILES["arch_a/training/__init__.py"] = "\nfrom .scale_optimizer import ScaleOptimizer\nfrom .galore2 import project_gradients_galore2\nfrom .spectron import spectral_renormalize_model\nfrom .mxfp8 import blockwise_mxfp8_quantize, blockwise_mxfp8_dequantize\n"
FILES["arch_a/training/scale_optimizer.py"] = "\nfrom __future__ import annotations\nfrom typing import Iterable, Optional\nimport torch\n\nclass ScaleOptimizer(torch.optim.Optimizer):\n    \"\"\"\n    SCALE optimizer:\n    - critical params: momentum + decoupled weight decay\n    - noncritical params: plain SGD step\n    This keeps optimizer state minimal.\n    \"\"\"\n    def __init__(self, params, lr_critical=1e-4, lr_noncritical=3e-4, momentum=0.9, weight_decay=0.0):\n        defaults = dict(lr_critical=lr_critical, lr_noncritical=lr_noncritical,\n                        momentum=momentum, weight_decay=weight_decay)\n        super().__init__(params, defaults)\n\n    @torch.no_grad()\n    def step(self, closure=None):\n        loss = None if closure is None else closure()\n        for group in self.param_groups:\n            lr = group[\"lr_critical\"] if group.get(\"critical\", False) else group[\"lr_noncritical\"]\n            wd = group.get(\"weight_decay\", 0.0)\n            mom = group.get(\"momentum\", 0.0)\n            for p in group[\"params\"]:\n                if p.grad is None:\n                    continue\n                grad = p.grad.float()\n                if grad.is_sparse:\n                    raise RuntimeError(\"ScaleOptimizer does not support sparse gradients.\")\n                if group.get(\"critical\", False):\n                    state = self.state[p]\n                    if \"momentum_buffer\" not in state:\n                        state[\"momentum_buffer\"] = torch.zeros_like(p, dtype=torch.float32)\n                    buf = state[\"momentum_buffer\"]\n                    buf.mul_(mom).add_(grad)\n                    update = buf\n                else:\n                    update = grad\n                if wd:\n                    p.mul_(1.0 - lr * wd)\n                p.add_(update.to(p.dtype), alpha=-lr)\n        return loss\n"
FILES["arch_a/training/galore2.py"] = "\nfrom __future__ import annotations\nfrom typing import Iterable, Optional\nimport torch\n\ndef _low_rank_project(grad: torch.Tensor, rank: int):\n    if grad.ndim != 2:\n        return grad\n    m, n = grad.shape\n    k = max(1, min(rank, min(m, n)))\n    if k >= min(m, n):\n        return grad\n    # Use a randomized low-rank approximation when available; fall back to SVD.\n    grad_f = grad.float()\n    try:\n        q = torch.randn(n, k, device=grad.device, dtype=torch.float32)\n        y = grad_f @ q\n        q2, _ = torch.linalg.qr(y, mode='reduced')\n        b = q2.transpose(0, 1) @ grad_f\n        return (q2 @ b).to(grad.dtype)\n    except Exception:\n        u, s, vh = torch.linalg.svd(grad_f, full_matrices=False)\n        return ((u[:, :k] * s[:k]) @ vh[:k, :]).to(grad.dtype)\n\n@torch.no_grad()\ndef project_gradients_galore2(model: torch.nn.Module, rank: int = 8, min_numel: int = 1024):\n    \"\"\"\n    Project large matrix gradients to a low-rank subspace.\n    Intended as a memory-conscious training-time hook.\n    \"\"\"\n    for p in model.parameters():\n        if p.grad is None:\n            continue\n        if p.grad.ndim == 2 and p.numel() >= min_numel:\n            p.grad = _low_rank_project(p.grad, rank)\n"
FILES["arch_a/training/spectron.py"] = "\nfrom __future__ import annotations\nimport torch\n\ndef _spectral_norm_estimate(w: torch.Tensor, power_iters: int = 1):\n    if w.ndim != 2:\n        return None\n    device = w.device\n    dtype = w.dtype\n    u = torch.randn(w.shape[0], 1, device=device, dtype=dtype)\n    for _ in range(power_iters):\n        v = torch.nn.functional.normalize(w.t() @ u, dim=0)\n        u = torch.nn.functional.normalize(w @ v, dim=0)\n    sigma = (u.t() @ w @ v).abs().squeeze()\n    return sigma\n\n@torch.no_grad()\ndef spectral_renormalize_model(model: torch.nn.Module, target_norm: float = 1.0, power_iters: int = 1):\n    \"\"\"\n    Spectron: keep matrix weights in a controlled spectral envelope.\n    \"\"\"\n    for p in model.parameters():\n        if p.ndim != 2:\n            continue\n        sigma = _spectral_norm_estimate(p.data.float(), power_iters=power_iters)\n        if sigma is None:\n            continue\n        sigma_val = float(sigma.detach().cpu())\n        if sigma_val > target_norm and sigma_val > 0:\n            scale = target_norm / sigma_val\n            p.data.mul_(scale)\n"
FILES["arch_a/training/mxfp8.py"] = "\nfrom __future__ import annotations\nimport torch\n\ndef blockwise_mxfp8_quantize(x: torch.Tensor, block_size: int = 32, eps: float = 1e-6):\n    \"\"\"\n    Simulated microscaling quantization.\n    Returns:\n        q: quantized tensor in float16 (for portability)\n        scale: per-block scale factors\n    \"\"\"\n    orig_shape = x.shape\n    flat = x.flatten()\n    n = flat.numel()\n    pad = (-n) % block_size\n    if pad:\n        flat = torch.cat([flat, flat.new_zeros(pad)])\n    blocks = flat.view(-1, block_size)\n    scale = blocks.abs().amax(dim=1, keepdim=True).clamp_min(eps)\n    q = torch.clamp((blocks / scale) * 127.0, -127, 127).round().to(torch.int8)\n    return q, scale.to(torch.float16), orig_shape\n\ndef blockwise_mxfp8_dequantize(q: torch.Tensor, scale: torch.Tensor, orig_shape, block_size: int = 32):\n    qf = q.to(torch.float32)\n    blocks = (qf / 127.0) * scale.to(torch.float32)\n    flat = blocks.flatten()[: int(torch.tensor(orig_shape).prod())]\n    return flat.view(*orig_shape)\n"
FILES["arch_a/quantization/__init__.py"] = "\nfrom .shadow_layer import ShadowResidualQuantizer\nfrom .polar_quant import PolarQuantizer, hadamard_rotate\n"
FILES["arch_a/quantization/shadow_layer.py"] = "\nfrom __future__ import annotations\nimport torch\n\nclass ShadowResidualQuantizer:\n    \"\"\"\n    1-bit residual correction for low-bit quantization.\n    This is a utility; it does not alter the main forward graph unless invoked.\n    \"\"\"\n    def __init__(self, bits: int = 4, block_size: int = 32):\n        self.bits = bits\n        self.block_size = block_size\n\n    def quantize(self, x: torch.Tensor):\n        orig_shape = x.shape\n        flat = x.flatten()\n        n = flat.numel()\n        pad = (-n) % self.block_size\n        if pad:\n            flat = torch.cat([flat, flat.new_zeros(pad)])\n        blocks = flat.view(-1, self.block_size)\n        scale = blocks.abs().amax(dim=1, keepdim=True).clamp_min(1e-6)\n        q = torch.clamp((blocks / scale) * (2 ** (self.bits - 1) - 1), -(2 ** (self.bits - 1)), 2 ** (self.bits - 1) - 1).round()\n        # 1-bit shadow residual sign map\n        residual = (blocks - (q / (2 ** (self.bits - 1) - 1)) * scale).sign().to(torch.int8)\n        return q.to(torch.int8), residual, scale.to(torch.float16), orig_shape\n\n    def dequantize(self, q: torch.Tensor, residual: torch.Tensor, scale: torch.Tensor, orig_shape):\n        qf = q.to(torch.float32)\n        rf = residual.to(torch.float32)\n        out = (qf / max(1, 2 ** (self.bits - 1) - 1)) * scale.to(torch.float32)\n        out = out + 0.125 * rf * scale.to(torch.float32)\n        flat = out.flatten()[: int(torch.tensor(orig_shape).prod())]\n        return flat.view(*orig_shape)\n"
FILES["arch_a/quantization/polar_quant.py"] = "\nfrom __future__ import annotations\nimport math\nimport torch\n\ndef _next_pow2(n: int) -> int:\n    return 1 << (n - 1).bit_length()\n\ndef hadamard_rotate(x: torch.Tensor):\n    \"\"\"\n    Fast Walsh-Hadamard transform on the last dimension.\n    Pads to next power of two if needed and trims back.\n    \"\"\"\n    orig_shape = x.shape\n    d = x.shape[-1]\n    n = _next_pow2(d)\n    if n != d:\n        pad = n - d\n        x = torch.nn.functional.pad(x, (0, pad))\n    y = x.clone()\n    h = 1\n    while h < n:\n        y = y.view(*y.shape[:-1], -1, 2, h)\n        a = y[..., 0, :].clone()\n        b = y[..., 1, :].clone()\n        y[..., 0, :] = a + b\n        y[..., 1, :] = a - b\n        y = y.view(*y.shape[:-3], -1)\n        h *= 2\n    y = y / math.sqrt(n)\n    return y[..., :d]\n\nclass PolarQuantizer:\n    def __init__(self, bits: int = 4, block_size: int = 32):\n        self.bits = bits\n        self.block_size = block_size\n\n    def quantize(self, x: torch.Tensor):\n        x_rot = hadamard_rotate(x.float())\n        flat = x_rot.flatten()\n        n = flat.numel()\n        pad = (-n) % self.block_size\n        if pad:\n            flat = torch.cat([flat, flat.new_zeros(pad)])\n        blocks = flat.view(-1, self.block_size)\n        scale = blocks.abs().amax(dim=1, keepdim=True).clamp_min(1e-6)\n        levels = 2 ** (self.bits - 1) - 1\n        q = torch.clamp((blocks / scale) * levels, -levels - 1, levels).round().to(torch.int8)\n        return q, scale.to(torch.float16), x.shape\n\n    def dequantize(self, q: torch.Tensor, scale: torch.Tensor, shape):\n        levels = 2 ** (self.bits - 1) - 1\n        flat = (q.to(torch.float32) / levels) * scale.to(torch.float32)\n        flat = flat.flatten()[: int(torch.tensor(shape).prod())]\n        return flat.view(*shape)\n"
FILES["arch_a/kernels/__init__.py"] = "\nfrom .dispatcher import KernelDispatcher\n"
FILES["arch_a/kernels/dispatcher.py"] = "\nfrom __future__ import annotations\nimport os\nimport torch\n\nclass KernelDispatcher:\n    \"\"\"\n    Unified accelerator fabric abstraction.\n    This does not magically compile to every backend, but it centralizes feature detection\n    and allows the model to select safe kernels per platform.\n    \"\"\"\n    def __init__(self):\n        self.backend = self.detect_backend()\n\n    def detect_backend(self):\n        if torch.cuda.is_available():\n            return \"cuda\"\n        try:\n            if hasattr(torch.backends, \"mps\") and torch.backends.mps.is_available():\n                return \"mps\"\n        except Exception:\n            pass\n        return \"cpu\"\n\n    @property\n    def supports_amp(self):\n        return self.backend in {\"cuda\", \"mps\"}\n\n    def maybe_compile(self, module):\n        try:\n            return torch.compile(module)\n        except Exception:\n            return module\n\n    def attention_fn(self):\n        return torch.nn.functional.scaled_dot_product_attention if hasattr(torch.nn.functional, \"scaled_dot_product_attention\") else None\n"
FILES["arch_a/model.py"] = "from __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import Any, Dict, List, Optional, Sequence, Tuple\nimport math\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\n\nfrom .config import ArchAConfig\nfrom .modules import ALGRBlock, ALGRController, NADDDecoder, RMSNorm\nfrom .modules.alpha_window import AlphaWindowState\nfrom .training.mxfp8 import blockwise_mxfp8_quantize, blockwise_mxfp8_dequantize\n\n@dataclass\nclass ArchAOutput:\n    logits: torch.Tensor\n    loss: Optional[torch.Tensor] = None\n    hidden_states: Optional[torch.Tensor] = None\n    ssm_states: Optional[List[AlphaWindowState]] = None\n    algr_meta: Optional[Any] = None\n    aux_losses: Optional[Dict[str, torch.Tensor]] = None\n\nclass ArchAModel(nn.Module):\n    def __init__(self, config: ArchAConfig):\n        super().__init__()\n        self.config = config\n        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)\n        self.emb_dropout = nn.Dropout(config.dropout)\n        self.layers = nn.ModuleList([\n            ALGRBlock(\n                d_model=config.d_model,\n                n_heads=config.n_heads,\n                n_kv_heads=config.n_kv_heads,\n                d_head=config.d_head,\n                ssm_state_dim=config.ssm_state_dim,\n                alpha_window=config.alpha_window,\n                d_ff=config.d_ff,\n                rope_theta=config.rope_theta,\n                norm_eps=config.norm_eps,\n                dropout=config.dropout,\n                use_bias=config.use_bias,\n                residual_fp32=config.use_fp32_residuals,\n                summary_dim=config.residual_summary_dim,\n            )\n            for _ in range(config.n_layers)\n        ])\n        self.final_norm = RMSNorm(config.d_model, config.norm_eps)\n        self.controller = ALGRController(\n            self.layers,\n            d_model=config.d_model,\n            max_loops=config.algr_max_loops,\n            confidence_threshold=config.algr_confidence_threshold,\n            temperature=config.algr_temperature,\n            device_map=None,\n        )\n        self.nadd_decoder = NADDDecoder(\n            d_model=config.d_model,\n            vocab_size=config.vocab_size,\n            steps=config.nadd_steps,\n            hidden_mult=config.nadd_hidden_mult,\n            dropout=config.dropout,\n            use_bias=config.use_bias,\n            noise_scale=config.nadd_noise_scale,\n        )\n        self.device_map: Optional[List[torch.device]] = None\n\n    def shard_to_devices(self, devices: Sequence[str | torch.device]):\n        devices = [torch.device(d) for d in devices]\n        self.device_map = devices\n        if len(devices) == 0:\n            return self\n        # simple contiguous partition\n        per = math.ceil(len(self.layers) / len(devices))\n        map_list = []\n        for i, layer in enumerate(self.layers):\n            dev = devices[min(i // per, len(devices) - 1)]\n            layer.to(dev)\n            map_list.append(dev)\n        self.controller.device_map = map_list\n        self.token_embedding.to(devices[0])\n        self.final_norm.to(devices[-1])\n        self.nadd_decoder.to(devices[-1])\n        # weight tying only when single device\n        if self.config.tie_embeddings and len(devices) == 1:\n            self.tie_weights()\n        return self\n\n    def tie_weights(self):\n        self.nadd_decoder.out_proj.weight = self.token_embedding.weight\n        return self\n\n    def init_states(self, batch_size: int, device: Optional[torch.device] = None):\n        states = []\n        for i, layer in enumerate(self.layers):\n            dev = device if device is not None else next(layer.parameters()).device\n            ssm = torch.zeros(batch_size, self.config.ssm_state_dim, device=dev, dtype=next(layer.parameters()).dtype)\n            summary = torch.zeros(batch_size, self.config.residual_summary_dim, device=dev, dtype=next(layer.parameters()).dtype)\n            states.append((ssm, summary))\n        return states\n\n    def forward_backbone(self, input_ids: torch.Tensor, ssm_states=None, training: bool = True):\n        emb_dev = self.token_embedding.weight.device\n        if input_ids.device != emb_dev:\n            input_ids = input_ids.to(emb_dev, non_blocking=True)\n        x = self.token_embedding(input_ids)\n        x = self.emb_dropout(x)\n        x, ssm_states, algr_meta = self.controller(self.layers, x, ssm_states, training=training)\n        x = self.final_norm(x)\n        return x, ssm_states, algr_meta\n\n    def forward(\n        self,\n        input_ids: torch.Tensor,\n        labels: Optional[torch.Tensor] = None,\n        ssm_states=None,\n        training_mode: str = \"hybrid\",\n        output_hidden_states: bool = False,\n        return_dict: bool = True,\n        **kwargs,\n    ):\n        hidden, ssm_states, algr_meta = self.forward_backbone(input_ids, ssm_states=ssm_states, training=self.training)\n        anchor = hidden.mean(dim=1)\n\n        # AR head uses tied embedding weights when possible, otherwise the decoder projection weight.\n        ar_weight = self.token_embedding.weight if (self.config.tie_embeddings and self.device_map is None) else self.nadd_decoder.out_proj.weight\n        ar_logits = F.linear(hidden, ar_weight)\n\n        refined_hidden, nadd_logits = self.nadd_decoder(hidden, anchor_state=anchor)\n        if training_mode == \"nadd\":\n            logits = nadd_logits\n            hidden_out = refined_hidden\n        elif training_mode == \"ar\":\n            logits = ar_logits\n            hidden_out = hidden\n        else:\n            logits = 0.6 * ar_logits + 0.4 * nadd_logits\n            hidden_out = refined_hidden\n\n        loss = None\n        aux_losses = {}\n        if labels is not None:\n            if labels.device != logits.device:\n                labels = labels.to(logits.device, non_blocking=True)\n            # causal next-token loss\n            shift_logits = logits[:, :-1].contiguous()\n            shift_labels = labels[:, 1:].contiguous()\n            loss = F.cross_entropy(\n                shift_logits.view(-1, shift_logits.size(-1)),\n                shift_labels.reshape(-1),\n                ignore_index=-100,\n            )\n            # lightly regularize halting entropy/loops\n            loop_penalty = torch.tensor(sum(algr_meta.loops) / max(1, len(algr_meta.loops)), device=loss.device, dtype=loss.dtype)\n            aux_losses[\"loop_penalty\"] = loop_penalty\n            loss = loss + 1e-4 * loop_penalty\n\n        out = ArchAOutput(\n            logits=logits,\n            loss=loss,\n            hidden_states=hidden_out if output_hidden_states else None,\n            ssm_states=ssm_states,\n            algr_meta=algr_meta,\n            aux_losses=aux_losses if aux_losses else None,\n        )\n        return out if return_dict else (out.logits, out.loss, out.hidden_states, out.ssm_states, out.algr_meta)\n\n    @torch.no_grad()\n    def generate_ar(self, input_ids: torch.Tensor, max_new_tokens: int = 32, temperature: float = 1.0):\n        self.eval()\n        seq = input_ids\n        ssm_states = None\n        for _ in range(max_new_tokens):\n            out = self.forward(seq, ssm_states=ssm_states, training_mode=\"ar\", return_dict=True)\n            logits = out.logits[:, -1] / max(temperature, 1e-6)\n            next_token = torch.multinomial(F.softmax(logits.float(), dim=-1), num_samples=1)\n            next_token = next_token.to(seq.device)\n            seq = torch.cat([seq, next_token], dim=1)\n            ssm_states = out.ssm_states\n        return seq\n\n    @torch.no_grad()\n    def generate_nadd(self, input_ids: torch.Tensor, steps: Optional[int] = None):\n        self.eval()\n        hidden, _, _ = self.forward_backbone(input_ids, training=False)\n        steps = steps or self.config.nadd_steps\n        refined = hidden\n        for i in range(steps):\n            refined, logits = self.nadd_decoder(refined, anchor_state=refined.mean(dim=1))\n        return logits\n\nclass ArchAForCausalLM(ArchAModel):\n    def __init__(self, config: ArchAConfig):\n        super().__init__(config)\n"
FILES["arch_a/tests.py"] = "from __future__ import annotations\nimport torch\nfrom arch_a import ArchAConfig, ArchAForCausalLM\nfrom arch_a.quantization import ShadowResidualQuantizer, PolarQuantizer\nfrom arch_a.training import (ScaleOptimizer, project_gradients_galore2,\n                              spectral_renormalize_model,\n                              blockwise_mxfp8_quantize, blockwise_mxfp8_dequantize)\n\ndef run():\n    cfg   = ArchAConfig.for_debug()\n    model = ArchAForCausalLM(cfg).train()\n    x = torch.randint(0, cfg.vocab_size, (1, 4))\n    y = torch.randint(0, cfg.vocab_size, (1, 4))\n    out = model(input_ids=x, labels=y, training_mode=\"hybrid\", output_hidden_states=True)\n    assert out.logits.shape == (1, 4, cfg.vocab_size)\n    assert out.loss is not None and torch.isfinite(out.loss)\n    assert out.hidden_states is not None\n    assert len(out.ssm_states) == cfg.n_layers\n    assert all(isinstance(v, int) for v in out.algr_meta.loops)\n    out_ar = model(input_ids=x, labels=y, training_mode=\"ar\")\n    assert torch.isfinite(out_ar.loss)\n    out_nd = model(input_ids=x, labels=y, training_mode=\"nadd\")\n    assert torch.isfinite(out_nd.loss)\n    qz = ShadowResidualQuantizer(bits=4)\n    q, r, s, shape = qz.quantize(torch.randn(3, 17))\n    assert qz.dequantize(q, r, s, shape).shape == (3, 17)\n    pq = PolarQuantizer(bits=4)\n    q2, s2, sh2 = pq.quantize(torch.randn(4, 16))\n    assert pq.dequantize(q2, s2, sh2).shape == (4, 16)\n    tensor = torch.randn(4, 64)\n    q3, s3, sh3 = blockwise_mxfp8_quantize(tensor, block_size=32)\n    assert blockwise_mxfp8_dequantize(q3, s3, sh3, block_size=32).shape == tensor.shape\n    params = list(model.parameters())\n    opt = ScaleOptimizer([\n        {\"params\": params[:4],  \"critical\": True,  \"lr_critical\": 1e-4, \"lr_noncritical\": 3e-4},\n        {\"params\": params[4:],  \"critical\": False, \"lr_critical\": 1e-4, \"lr_noncritical\": 3e-4},\n    ])\n    out.loss.backward()\n    project_gradients_galore2(model, rank=2)\n    spectral_renormalize_model(model, target_norm=2.0)\n    opt.step()\n    opt.zero_grad(set_to_none=True)\n    model.eval()\n    gen = model.generate_ar(x, max_new_tokens=4)\n    assert gen.shape == (1, 8)\n    print(\"\u2713 All tests passed\")\n\nif __name__ == \"__main__\":\n    run()\n"
for rel, content in FILES.items():
    with open(os.path.join(BASE, rel), 'w') as f: f.write(content)
if BASE not in sys.path: sys.path.insert(0, BASE)
for k in list(sys.modules.keys()):
    if k.startswith('arch_a'): del sys.modules[k]
from arch_a import ArchAConfig, ArchAForCausalLM
print(f'✓ arch_a v0.3.2 ready  ({len(FILES)} files)')

## Cell 3 — Test Suite

Basic correctness: shapes, losses finite, gradients flow.

In [ ]:
import subprocess, sys, os
env = os.environ.copy(); env['PYTHONPATH'] = '/kaggle/working'
r = subprocess.run([sys.executable,'/kaggle/working/arch_a/tests.py'],
    capture_output=True, text=True, cwd='/kaggle/working', env=env)
print(r.stdout)
if r.returncode != 0:
    bad=[l for l in r.stderr.splitlines() if 'Error' in l or 'Traceback' in l]
    print('\n'.join(bad))

## Cell 4 — Build 1B Model (fp32, single device)

In [ ]:
from arch_a import ArchAConfig, ArchAForCausalLM
vram_gb = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
cfg = ArchAConfig.for_1b() if vram_gb >= 14 else ArchAConfig.for_500m()
# fp32 weights — DO NOT call .half(). autocast handles fp16 activations during forward.
model = ArchAForCausalLM(cfg).to(device)
n = sum(p.numel() for p in model.parameters())
print(f'Params: {n/1e6:.1f}M  |  dtype: {next(model.parameters()).dtype}  |  device: {device}')
vram()

## Cell 5 — Architecture Validation

Checks the key properties that determine whether pre-training will succeed or fail.

In [ ]:
# ── 5a. Forward pass ──────────────────────────────────────────────
B, L = 2, min(64, cfg.alpha_window)
x = torch.randint(0, cfg.vocab_size, (B, L), device=device)
y = torch.randint(0, cfg.vocab_size, (B, L), device=device)
model.eval()
with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
    out = model(input_ids=x, labels=y, training_mode='hybrid')
print('── Forward pass ──────────────────────')
print(f'Logits : {tuple(out.logits.shape)}')
print(f'Loss   : {out.loss.item():.4f}  (expect ~log(vocab)={torch.log(torch.tensor(cfg.vocab_size)).item():.1f})')
print(f'NaN    : {torch.isnan(out.logits.float()).any().item()}')
print(f'ALGR loops per layer: {out.algr_meta.loops}')

# ── 5b. Loss is near random baseline ──────────────────────────────
random_baseline = torch.log(torch.tensor(float(cfg.vocab_size))).item()
loss_ratio = out.loss.item() / random_baseline
print(f'\nLoss/random_baseline = {loss_ratio:.3f}  (expect 0.9–1.1 for untrained model)')
assert 0.5 < loss_ratio < 2.0, f'Loss far from random baseline — architecture bug! ratio={loss_ratio}'
print('✓ Loss sanity check passed')

# ── 5c. ALGR actually loops (confidence gate working) ─────────────
max_loops = max(out.algr_meta.loops)
print(f'\nMax ALGR loops: {max_loops}  (expect >=1)')
assert max_loops >= 1, 'ALGR never looped — confidence gate broken'
print('✓ ALGR loop check passed')

# ── 5d. SSM states are non-zero (SSM is computing) ────────────────
ssm_norms = [s[0].norm().item() if s is not None else 0 for s in out.ssm_states]
print(f'\nSSM state norms: {[f"{v:.3f}" for v in ssm_norms[:4]]}...')
assert any(v > 0 for v in ssm_norms), 'All SSM states zero — SSM not computing'
print('✓ SSM state check passed')

# ── 5e. NADD logits differ from AR logits (NADD is active) ────────
with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
    out_ar   = model(input_ids=x, training_mode='ar')
    out_nadd = model(input_ids=x, training_mode='nadd')
diff = (out_ar.logits.float() - out_nadd.logits.float()).abs().mean().item()
print(f'\nAR vs NADD logit diff: {diff:.4f}  (expect >0 — NADD must differ from AR)')
assert diff > 0, 'AR and NADD logits identical — NADD decoder not working'
print('✓ NADD activity check passed')
print('\n✓ ALL ARCHITECTURE CHECKS PASSED')
vram()

## Cell 6 — Backward Pass & Gradient Health

In [ ]:
torch.cuda.empty_cache(); gc.collect()
model.train()
scaler = torch.amp.GradScaler('cuda')
with torch.amp.autocast('cuda', dtype=torch.float16):
    out2 = model(input_ids=x, labels=y, training_mode='ar')
scaler.scale(out2.loss).backward()

grads = {n: p.grad for n, p in model.named_parameters() if p.grad is not None}
total  = len(list(model.parameters()))
w_grad = len(grads)
nans   = sum(1 for g in grads.values() if torch.isnan(g).any())
zeros  = sum(1 for g in grads.values() if g.abs().max() == 0)
print(f'Params total      : {total}')
print(f'Params with grads : {w_grad}')
print(f'NaN gradients     : {nans}   {"✓" if nans==0 else "✗ PROBLEM"}')
print(f'Zero gradients    : {zeros}  {"✓" if zeros<w_grad*0.5 else "✗ DEAD NEURONS"}')

# Check grad norms per component
components = {'ssm/decay':[], 'attention':[], 'ffn':[], 'nadd':[], 'algr_halt':[]}
for n, g in grads.items():
    norm = g.float().norm().item()
    if 'decay' in n: components['ssm/decay'].append(norm)
    elif any(k in n for k in ['q_proj','k_proj','v_proj','o_proj']): components['attention'].append(norm)
    elif 'mlp' in n or 'ff' in n: components['ffn'].append(norm)
    elif 'nadd' in n or 'refiner' in n: components['nadd'].append(norm)
    elif 'halt' in n: components['algr_halt'].append(norm)
print()
print('Gradient norm by component:')
for comp, norms in components.items():
    if norms: print(f'  {comp:15s}: mean={sum(norms)/len(norms):.4f}  max={max(norms):.4f}')

assert nans == 0, 'NaN gradients detected — training will diverge!'
print('\n✓ Gradient health check passed')

## Cell 7 — Training Stability (5 steps, loss must decrease)

In [ ]:
from arch_a.training import (ScaleOptimizer, project_gradients_galore2, spectral_renormalize_model)
torch.cuda.empty_cache(); gc.collect()
model.train()

critical_kw = ('decay','gate','norm','halt_head','summary_proj','anchor_proj')
crit, noncrit = [], []
for name, p in model.named_parameters():
    (crit if any(k in name for k in critical_kw) else noncrit).append(p)
opt = ScaleOptimizer([
    {'params': crit,    'critical': True,  'lr_critical': cfg.scale_critical_lr, 'lr_noncritical': cfg.scale_noncritical_lr},
    {'params': noncrit, 'critical': False, 'lr_critical': cfg.scale_critical_lr, 'lr_noncritical': cfg.scale_noncritical_lr},
])
scaler2 = torch.amp.GradScaler('cuda')

losses = []
print('Step | Loss   | ALGR loops')
print('-----|--------|------------------')
for step in range(5):
    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda', dtype=torch.float16):
        out3 = model(input_ids=x, labels=y, training_mode='hybrid')
    if not torch.isfinite(out3.loss): print('✗ Non-finite loss — training broken'); break
    scaler2.scale(out3.loss).backward()
    scaler2.unscale_(opt)
    project_gradients_galore2(model, rank=cfg.galore_rank)
    spectral_renormalize_model(model, target_norm=cfg.spectron_target_norm, power_iters=cfg.spectron_power_iters)
    scaler2.step(opt); scaler2.update()
    l = out3.loss.item(); losses.append(l)
    print(f'  {step+1}  | {l:.4f} | {out3.algr_meta.loops}')

trend = losses[-1] < losses[0]
print(f'\nLoss: {losses[0]:.4f} → {losses[-1]:.4f}  [Decreasing: {trend}]')
print(f'Training stack: {"✓ STABLE" if trend else "⚠ NOT DECREASING (normal for 5 random steps)"}' )
vram()

## Cell 8 — Save Portable Checkpoint

This file will be uploaded to AMD Developer Cloud to prove hardware agnosticism.

In [ ]:
model.eval()
CKPT = '/kaggle/working/arch_a_alpha_agi.pt'
torch.save({'config': cfg.to_dict(), 'state_dict': model.state_dict(),
            'trained_on': 'NVIDIA T4 (CUDA)', 'kaggle_losses': losses}, CKPT)
sz = Path(CKPT).stat().st_size/1e6
print(f'Saved: {CKPT}  ({sz:.1f} MB)')
# CPU reload test
chk = torch.load(CKPT, map_location='cpu', weights_only=True)
m2  = ArchAForCausalLM(ArchAConfig.from_dict(chk['config']))
m2.load_state_dict(chk['state_dict'], strict=True)
print(f'CPU reload: ✓  ({sum(p.numel() for p in m2.parameters())/1e6:.1f}M params)')
print('\n→ Download arch_a_alpha_agi.pt and upload to AMD Developer Cloud')
print('→ Run ARCH_A_AMD_PORTABILITY.ipynb there')